<a href="https://colab.research.google.com/github/tracyhann/bsl-data-tools/blob/main/redcap-cleaner/REDCap_cleaner_step_by_step.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REDCap Cleaner

Run each cell from top to bottom. Click the ▶ to execute each step/section.

# 🏃 Setup (ALWAYS RUN THIS FIRST!)


*   Click the ▶ button below once, next to **[↪ 3 cells hidden]**, to run all setup codes.
*   Alternatively, run each of the cell: Cleaner engine, Helper functions


In [ ]:
# @title Cleaner engine {"display-mode":"form"}
from __future__ import annotations

from dataclasses import dataclass
from io import BytesIO
import math
import re
from typing import Iterable

import pandas as pd

import warnings

warnings.simplefilter("ignore", pd.errors.PerformanceWarning)



SUBJECT_COLUMN_CANDIDATES = (
    "subid",
    "subject_id",
    "subject",
    "participant_id",
    "record_id",
    "record_subject_id",
)
RESERVED_OUTPUT_COLUMNS = {"IRB", "subid", "raw_subid"}


@dataclass(frozen=True)
class ColumnRule:
    original: str
    keep: bool = True
    rename: str = ""


@dataclass
class CleanResult:
    cleaned: pd.DataFrame
    variable_map: dict[str, pd.DataFrame]
    outliers: pd.DataFrame
    exclusions: pd.DataFrame


@dataclass(frozen=True)
class _Identifier:
    raw: object
    raw_text: str
    compact: str
    subid: str | None
    irb: str | None
    is_outlier: bool = False
    outlier_reason: str = ""


def sanitize_irbs(values: Iterable[object]) -> list[str]:
    """Return non-empty IRB aliases, preserving first-seen order.

    Numeric aliases are normalized to digit-only strings, e.g. "IRB-12345" -> "12345".
    Keyword aliases are normalized to uppercase alphanumeric tokens, e.g. "ol" -> "OL".
    """
    seen: set[str] = set()
    clean: list[str] = []
    for value in values:
        text = _cell_text(value)
        if not text:
            continue
        for part in re.split(r"[,;/|]+", text):
            token = _normalize_irb_alias(part)
            if token and token not in seen:
                clean.append(token)
                seen.add(token)
    return clean


def detect_subject_column(columns: Iterable[str]) -> str | None:
    normalized = {_normalize_column_name(column): str(column) for column in columns}
    for candidate in SUBJECT_COLUMN_CANDIDATES:
        normalized_candidate = _normalize_column_name(candidate)
        if normalized_candidate in normalized:
            return normalized[normalized_candidate]
    return None


def _normalize_column_name(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).strip().lower())




def _cell_text(value: object) -> str:
    """Convert table/IRB cell values to text without corrupting integer-like floats."""
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(value, float):
        if math.isnan(value):
            return ""
        if value.is_integer():
            return str(int(value))
    return str(value).strip()


def _normalize_irb_alias(value: object) -> str:
    """Normalize one IRB alias from user input.

    Examples:
        12345 -> "12345"
        "IRB-12345" -> "12345"
        "OL" -> "OL"
        "irb_OL" -> "OL"
    """
    text = _cell_text(value)
    if not text:
        return ""
    text = re.sub(r"(?i)^\s*irb[\s_-]*", "", text.strip())

    # Preserve numeric IRBs safely, including values that arrived as "12345.0".
    if re.fullmatch(r"\d+(?:\.0+)?", text):
        return str(int(float(text))) if "." in text else text

    token = re.sub(r"[^A-Za-z0-9]+", "", text)
    if not token:
        return ""
    if token.isdigit():
        return token
    if token.isalpha():
        return token.upper()

    # Mixed aliases are suspicious as IRB definitions. Prefer numeric content if present.
    # This keeps old-ish behavior for entries like "IRB #12345".
    digits = re.sub(r"\D+", "", token)
    return digits or token.upper()


def _is_numeric_irb_alias(alias: str) -> bool:
    return bool(re.fullmatch(r"\d+", alias))


def _irb_alias_pattern(alias: str) -> re.Pattern[str]:
    if _is_numeric_irb_alias(alias):
        return re.compile(rf"(?<!\d)(?:irb[\s_-]*)?{re.escape(alias)}(?!\d)", re.I)
    return re.compile(
        rf"(?<![A-Za-z0-9])(?:irb[\s_-]*)?{re.escape(alias)}(?=\d|[^A-Za-z0-9]|$)",
        re.I,
    )


def _keyword_subject_pattern(keyword: str) -> re.Pattern[str]:
    return re.compile(
        rf"(?<![A-Za-z0-9])(?:irb[\s_-]*)?{re.escape(keyword)}[\s_-]*(\d+)",
        re.I,
    )


def _numeric_irb_subject_pattern(irb: str) -> re.Pattern[str]:
    """Detect subject digits after a numeric IRB alias.

    Examples:
        12345s001 -> 001
        IRB12345s001 -> 001
        12345_001 -> 001
        IRB-12345-sub001 -> 001

    The pattern intentionally does not match 12345001 because that is ambiguous:
    it may be one long identifier rather than IRB 12345 + subject 001.
    """
    return re.compile(
        rf"(?<!\d)(?:irb[\s_-]*)?{re.escape(irb)}(?:[\s_-]*(?:sub|s)[\s_-]*|[_-]+)(\d+)(?!\d)",
        re.I,
    )


def _format_subid(digits: str) -> str | None:
    if not digits:
        return None
    return f"s{digits.zfill(3) if len(digits) < 3 else digits}"


def _unique_preserve_order(values: Iterable[str]) -> list[str]:
    seen: set[str] = set()
    out: list[str] = []
    for value in values:
        if value and value not in seen:
            out.append(value)
            seen.add(value)
    return out


def read_table(filename: str, content: bytes) -> pd.DataFrame:
    suffix = filename.lower().rsplit(".", 1)[-1] if "." in filename else ""
    buffer = BytesIO(content)
    if suffix == "csv":
        return pd.read_csv(buffer)
    if suffix in {"xls", "xlsx"}:
        return pd.read_excel(buffer)
    raise ValueError("Upload must be a .csv, .xls, or .xlsx file.")

def _source_row_data(df, source_index) -> dict[str, object]:
        return {
            f"source__{col}": df.loc[source_index, col]
            for col in df.columns
        }

def clean_table(
    raw: pd.DataFrame,
    *,
    subject_column: str,
    irbs: Iterable[object],
    column_rules: Iterable[ColumnRule],
    include_outlier_raw_values: Iterable[object] = (),
    drop_empty_rows: bool = False,
) -> CleanResult:
    if subject_column not in raw.columns:
        raise ValueError(f"Subject identifier column not found: {subject_column}")

    irb_values = sanitize_irbs(irbs)
    data = raw.copy()

    rows_to_keep: list[bool] = []
    cleaned_ids: list[str | None] = []
    extracted_irbs: list[str | None] = []
    raw_subids: list[object] = []
    outlier_rows: list[dict[str, object]] = []
    excluded_rows: list[dict[str, object]] = []


    if drop_empty_rows:
        non_empty_fraction = data.notna().mean(axis=1)
        dropped_empty_mask = non_empty_fraction < 0.30

        for source_index in data.index[dropped_empty_mask]:
            excluded_rows.append(
                {
                    "source_row": _source_row_number(source_index),
                    "raw_subid": data.loc[source_index, subject_column]
                    if subject_column in data.columns
                    else "",
                    "cleaned_subid": "",
                    "included": False,
                    "reason": "excluded from cleaned file; row >70% empty",
                    "non_empty_fraction": non_empty_fraction.loc[source_index],
                    **_source_row_data(data, source_index),
                }
            )

        data = data.loc[~dropped_empty_mask].copy()

    identifiers = _classify_identifiers(data[subject_column], irb_values)
    include_outliers = {str(value) for value in include_outlier_raw_values}

    for source_index, ident in zip(data.index, identifiers):
        manually_included = ident.raw_text in include_outliers
        excluded = ident.is_outlier and not manually_included

        rows_to_keep.append(not excluded)
        cleaned_ids.append(ident.subid if ident.subid is not None else ident.compact)
        extracted_irbs.append(ident.irb)
        raw_subids.append(ident.raw)

        source_row_data = _source_row_data(data, source_index)

        if ident.is_outlier:
            outlier_rows.append(
                {
                    "source_row": _source_row_number(source_index),
                    "raw_subid": ident.raw,
                    "cleaned_subid": "",
                    "included": bool(manually_included),
                    "reason": ident.outlier_reason,
                    **source_row_data,
                }
            )

        if excluded:
            excluded_rows.append(
                {
                    "source_row": _source_row_number(source_index),
                    "raw_subid": ident.raw,
                    "cleaned_subid": "",
                    "included": bool(manually_included),
                    "reason": "excluded from cleaned file; manual reviewed",
                    **source_row_data,
                }
            )

    base = pd.DataFrame(
        {
            "IRB": extracted_irbs,
            "subid": cleaned_ids,
            "raw_subid": raw_subids,
        },
        index=data.index,
    )

    mapped_columns, column_map = _apply_column_rules(data, subject_column, column_rules)
    cleaned = pd.concat([base, mapped_columns], axis=1)
    cleaned = cleaned.loc[rows_to_keep].reset_index(drop=True)
    cleaned = _none_for_missing(cleaned)

    variable_map = {
        "column_map": pd.DataFrame(column_map),
        "excluded_subjects": pd.DataFrame(excluded_rows),
        "outlier_review": pd.DataFrame(outlier_rows),
    }
    return CleanResult(
        cleaned=cleaned,
        variable_map=variable_map,
        outliers=variable_map["outlier_review"],
        exclusions = variable_map["excluded_subjects"]
    )


def dataframes_to_excel_bytes(sheets: dict[str, pd.DataFrame]) -> bytes:
    buffer = BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        for sheet_name, dataframe in sheets.items():
            dataframe.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    return buffer.getvalue()


def _classify_identifiers(values: pd.Series, irbs: list[str]) -> list[_Identifier]:
    initial = [_clean_identifier(value, irbs) for value in values]
    classified: list[_Identifier] = []

    for ident in initial:
        reasons: list[str] = []
        raw_identifier = ident.raw_text.strip()

        if irbs and not ident.irb:
            reasons.append("identifier does not match any known IRB alias")

        if raw_identifier and len(raw_identifier) < 4:
            reasons.append("raw identifier has fewer than 4 characters")

        if not ident.subid:
            reasons.append("identifier does not match s### or numeric format after cleaning")
        elif not re.fullmatch(r"s\d{3}", ident.subid):
            reasons.append("cleaned identifier does not match expected s### format")

        if reasons:
            classified.append(
                _Identifier(
                    raw=ident.raw,
                    raw_text=ident.raw_text,
                    compact=ident.compact,
                    subid=ident.subid,
                    irb=ident.irb,
                    is_outlier=True,
                    outlier_reason="; ".join(reasons),
                )
            )
        else:
            classified.append(ident)

    return classified



def _clean_identifier(value: object, irbs: list[str]) -> _Identifier:
    raw_text = _cell_text(value)
    working = raw_text.strip()
    compact = re.sub(r"[-_\s]+", "", working).strip()
    compact = re.sub(r"[^A-Za-z0-9]+", "", compact)

    if not working:
        return _Identifier(raw=value, raw_text=raw_text, compact=compact, subid=None, irb=None)

    matched_irbs = [irb for irb in irbs if _irb_alias_pattern(irb).search(working)]
    matched_irbs = _unique_preserve_order(matched_irbs)
    found_irb = matched_irbs[0] if matched_irbs else None

    reasons: list[str] = []
    if len(matched_irbs) > 1:
        reasons.append(f"multiple IRB aliases matched: {matched_irbs}")

    subid_candidates: list[str] = []

    if found_irb:
        # Rule A: if the matched IRB alias is a keyword, detect digits immediately after it.
        # Examples: OL001, OL_001, OL-001, OL 001, IRB_OL_001.
        for irb in matched_irbs:
            if _is_numeric_irb_alias(irb):
                # Numeric IRB followed by subject ID.
                # Examples: 12345s001, IRB12345s001, 12345_001, IRB-12345-sub001.
                subid_candidates.extend(
                    match.group(1)
                    for match in _numeric_irb_subject_pattern(irb).finditer(working)
                )
            else:
                subid_candidates.extend(
                    match.group(1)
                    for match in _keyword_subject_pattern(irb).finditer(working)
                )

        # Rule B (PATCHED): only use trailing _### / -### as fallback when Rule A found nothing.
        # Prevents visit/run suffixes like "_2" in "64771_s067_2" from being treated as a second subject id.
        if not subid_candidates:
            tail_match = re.search(r"[_-](\d+)(?:\.[A-Za-z0-9]+)?\s*$", working)
            if tail_match:
                subid_candidates.append(tail_match.group(1))

        subid_candidates = _unique_preserve_order(subid_candidates)
        if len(subid_candidates) > 1:
            reasons.append(f"multiple subject IDs detected: {subid_candidates}")

        subid = _format_subid(subid_candidates[0]) if subid_candidates else None
        return _Identifier(
            raw=value,
            raw_text=raw_text,
            compact=compact,
            subid=subid,
            irb=found_irb,
            is_outlier=bool(reasons),
            outlier_reason="; ".join(reasons),
        )

    # No IRB alias matched (PATCHED): be strict.
    # We DO NOT extract arbitrary digits anymore (drops things like "MDD_916" when only IRB is 64771).
    match = re.fullmatch(r"(?i)(?:sub)?s[-_ ]?(\d+)(?:[A-Za-z])?", compact)
    subid = None
    if match:
        subid = _format_subid(match.group(1))
    else:
        # Also accept explicit markers in the raw string, e.g. "..._s067_2" or "sub067"
        marker_match = re.search(r"(?i)(?:^|[^A-Za-z0-9])(sub|s)[-_ ]*(\d+)\b", working)
        if marker_match:
            subid = _format_subid(marker_match.group(2))

    return _Identifier(raw=value, raw_text=raw_text, compact=compact, subid=subid, irb=None)




def _apply_column_rules(
    data: pd.DataFrame,
    subject_column: str,
    column_rules: Iterable[ColumnRule],
) -> tuple[pd.DataFrame, list[dict[str, object]]]:
    output = pd.DataFrame(index=data.index)
    map_rows: list[dict[str, object]] = []
    used_names = set(RESERVED_OUTPUT_COLUMNS)

    rules_by_column = {rule.original: rule for rule in column_rules}
    for column in data.columns:
        rule = rules_by_column.get(column, ColumnRule(original=column))
        role = "subject_id" if column == subject_column else ""
        clean_name = _dedupe_column_name((rule.rename or column).strip(), used_names)
        if rule.keep and column != subject_column:
            output[clean_name] = data[column]
            used_names.add(clean_name)
        map_rows.append(
            {
                "original_column": column,
                "clean_column": clean_name if rule.keep and column != subject_column else "",
                "keep": bool(rule.keep and column != subject_column),
                "role": role,
            }
        )
    return output, map_rows


def _dedupe_column_name(name: str, used_names: set[str]) -> str:
    base = name or "unnamed_column"
    candidate = base
    counter = 2
    while candidate in used_names:
        candidate = f"{base}_{counter}"
        counter += 1
    return candidate


def _source_row_number(index: object) -> object:
    if isinstance(index, int):
        return index + 2
    return index


def _none_for_missing(dataframe: pd.DataFrame) -> pd.DataFrame:
    return dataframe.astype(object).where(pd.notna(dataframe), None)


In [ ]:
# @title Helper functions {"display-mode":"form"}
from __future__ import annotations

from pathlib import Path
import re

import ipywidgets as widgets
from IPython.display import FileLink, display

try:
    from google.colab import files as colab_files
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    IN_COLAB = True
except Exception:
    colab_files = None
    IN_COLAB = False


uploaded_filename = ""
uploaded_dataframe = None
configure_ui = None
timepoint_ui = None
review_result = None
clean_result = None
include_outliers = []
timepoint_dictionary = None
cleaned_path = None
map_path = None


VISIT_COLUMN_CANDIDATES = (
    "visit",
    "visits",
    "visit_number",
    "visitnumber",
    "visit_name",
    "event_name",
    "redcap_event_name",
    "timepoint",
    "time_point",
    "administered during visit number",
)
TIMEPOINT_DICTIONARY_COLUMNS = ["visit_column", "source_visit", "timepoint"]


def require_uploaded_dataframe():
    if uploaded_dataframe is None:
        raise ValueError("Run step 1 to upload a REDCap CSV/XLSX export first.")
    return uploaded_dataframe


def require_configure_ui():
    if configure_ui is None:
        raise ValueError("Run step 2 to configure cleaning first.")
    return configure_ui


def run_clean(include_outlier_values=None):
    dataframe = require_uploaded_dataframe()
    config = require_configure_ui()
    return clean_table(
        dataframe,
        subject_column=config.subject_column,
        irbs=config.irbs,
        column_rules=config.column_rules(),
        include_outlier_raw_values=include_outlier_values or [],
        drop_empty_rows=config.drop_empty_rows.value,
    )


def output_stem(filename: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", Path(filename).stem) or "redcap_export"


def detect_visit_column(columns):
    normalized = {_normalize_column_name(column): str(column) for column in columns}
    for candidate in VISIT_COLUMN_CANDIDATES:
        normalized_candidate = _normalize_column_name(candidate)
        if normalized_candidate in normalized:
            return normalized[normalized_candidate]
    for column in columns:
        normalized_column = _normalize_column_name(column)
        if "visit" in normalized_column or "timepoint" in normalized_column:
            return str(column)
    return None


def canonicalize_timepoint(value):
    if value is None or pd.isna(value):
        return ""

    text = re.sub(r"\s+", " ", str(value)).strip()
    if not text:
        return ""

    # Handles:
    # visit2, visit 2, visit_2, visit-2
    # v2, v_2, v-02
    # baseline_visit_2, visit_2_baseline, visit_2_arm_1
    visit_match = re.search(
        r"(?i)(?<![A-Za-z0-9])(?:visit|v)[\s_-]*0*(\d+)(?!\d)",
        text,
    )
    if visit_match:
        return f"V{int(visit_match.group(1))}"

    numeric_match = re.fullmatch(r"0*(\d+)(?:\.0+)?", text)
    if numeric_match:
        return f"V{int(numeric_match.group(1))}"

    return text


def empty_timepoint_dictionary():
    return pd.DataFrame(columns=TIMEPOINT_DICTIONARY_COLUMNS)


def apply_timepoint_mapping(clean_result, mapping_ui):
    if mapping_ui is None:
        return empty_timepoint_dictionary()

    dictionary = mapping_ui.timepoint_dictionary()
    visit_column = mapping_ui.selected_visit_column
    if not visit_column or dictionary.empty:
        return dictionary

    column_map = clean_result.variable_map.get("column_map", pd.DataFrame())
    match = column_map[column_map["original_column"].astype(str).eq(str(visit_column))]
    if match.empty or not bool(match.iloc[0].get("keep", False)):
        raise ValueError("Selected visits column must be kept in step 2 before timepoint mapping can be applied.")

    clean_column = match.iloc[0].get("clean_column", "")
    if not clean_column or clean_column not in clean_result.cleaned.columns:
        raise ValueError("Selected visits column is not present in the cleaned output.")

    mapping = {
        _timepoint_value_key(row["source_visit"]): row["timepoint"]
        for _, row in dictionary.iterrows()
        if _timepoint_value_key(row["source_visit"])
    }
    clean_result.cleaned = clean_result.cleaned.copy()
    clean_result.cleaned[clean_column] = clean_result.cleaned[clean_column].map(
        lambda value: mapping.get(_timepoint_value_key(value), value)
    )
    return dictionary


def _timepoint_value_key(value):
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value).strip()


class TimepointMappingUI:
    def __init__(self, dataframe, config):
        self.dataframe = dataframe
        self.config = config
        self.mapping_controls = []
        self.visit_dropdown = widgets.Dropdown(
            description="Visits column",
            options=[""] + [str(column) for column in dataframe.columns],
            layout=widgets.Layout(width="100%"),
            style={"description_width": "130px"},
        )
        detected = detect_visit_column(dataframe.columns)
        if detected:
            self.visit_dropdown.value = detected
        self.mapping_table = widgets.GridBox(
            [],
            layout=widgets.Layout(
                width="100%",
                max_width="none",
                max_height="420px",
                overflow="auto",
                border="1px solid #c9d6dd",
                padding="0",
                grid_template_columns="minmax(260px, 1fr) minmax(150px, 220px) minmax(190px, 280px)",
                align_items="stretch",
            ),
        )
        self.visit_dropdown.observe(self._on_visit_change, names="value")
        self._render_mapping_controls()
        self.ui = widgets.VBox(
            [
                widgets.HTML("<h3>Review visit/timepoint mapping</h3>"),
                widgets.HTML("Unique values from the selected visits column will be canonicalized in the cleaned file. Edit the final timepoint values before running step 5 if needed."),
                self.visit_dropdown,
                self.mapping_table,
            ],
            layout=widgets.Layout(width="100%", max_width="none"),
        )

    @property
    def selected_visit_column(self):
        return self.visit_dropdown.value

    def timepoint_dictionary(self):
        if not self.selected_visit_column:
            return empty_timepoint_dictionary()
        rows = []
        for raw_value, proposed_value, canonical_text in self.mapping_controls:
            rows.append(
                {
                    "visit_column": self.selected_visit_column,
                    "source_visit": raw_value,
                    "timepoint": canonical_text.value.strip() or proposed_value,
                }
            )
        return pd.DataFrame(rows, columns=TIMEPOINT_DICTIONARY_COLUMNS)

    def _on_visit_change(self, change):
        self._render_mapping_controls()

    def _render_mapping_controls(self):
        self.mapping_controls = []
        if not self.selected_visit_column:
            self.mapping_table.children = (
                widgets.HTML("<b>No visits column selected.</b>"),
                widgets.HTML(""),
                widgets.HTML(""),
            )
            return

        values = []
        seen = set()
        for value in self.dataframe[self.selected_visit_column].drop_duplicates().tolist():
            key = _timepoint_value_key(value)
            if not key or key in seen:
                continue
            seen.add(key)
            values.append(value)

        cells = [
            widgets.HTML("<b>Raw visit value</b>"),
            widgets.HTML("<b>Proposed</b>"),
            widgets.HTML("<b>Use in cleaned file</b>"),
        ]
        if not values:
            cells.extend([
                widgets.HTML("No non-empty visit values found."),
                widgets.HTML(""),
                widgets.HTML(""),
            ])
        for raw_value in values:
            proposed = canonicalize_timepoint(raw_value)
            canonical_text = widgets.Text(value=proposed, layout=widgets.Layout(width="100%"))
            raw_label = widgets.HTML(f"<code>{raw_value}</code>")
            proposed_label = widgets.HTML(f"<code>{proposed}</code>")
            self.mapping_controls.append((raw_value, proposed, canonical_text))
            cells.extend([raw_label, proposed_label, canonical_text])
        self.mapping_table.children = tuple(cells)


def cleaned_data_summary(clean_result):
    df = clean_result.cleaned.copy()

    # Normalize subid
    df["subid_norm"] = (
        df["subid"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    # Extract numeric part: s001 -> 1
    df["subid_num"] = df["subid_norm"].str.extract(r"s?(\d+)").astype(float)

    # ---- 1. Unique subids ----
    unique_subids = sorted(
        df["subid_norm"].dropna().unique(),
        key=lambda x: int(pd.Series([x]).str.extract(r"s?(\d+)").iloc[0, 0])
        if pd.Series([x]).str.extract(r"s?(\d+)").iloc[0, 0] is not None
        else x
    )

    print(f"Number of unique subids: {len(unique_subids)}")

    # ---- 2. Missing continuous IDs ----
    observed_nums = sorted(df["subid_num"].dropna().astype(int).unique())

    min_id = min(observed_nums)
    max_id = max(observed_nums)

    expected_nums = set(range(min_id, max_id + 1))
    missing_nums = sorted(expected_nums - set(observed_nums))

    missing_subids = [f"s{x:03d}" for x in missing_nums]

    print(f"\nObserved numeric range: s{min_id:03d} to s{max_id:03d}")

    if missing_subids:
        print("Missing subids:")
        print(missing_subids)
    else:
        print("No missing subids within observed numeric range.")

    # ---- 3. Row counts per subid ----
    row_counts = (
        df.groupby("subid_norm")
        .size()
        .reset_index(name="n_rows")
        .sort_values(
            by="subid_norm",
            key=lambda col: col.str.extract(r"s?(\d+)")[0].astype(float)
        )
    )

    print("\nRows per subid:")
    display(data_table.DataTable(row_counts,num_rows_per_page=10))


# ⛳ REDCap cleaner step-by-step

*   Click the ▶ button below to execute ***each step***



In [ ]:
# @title 1. 📄 Upload REDCap export {"vertical-output":true,"display-mode":"form"}

global uploaded_filename, uploaded_dataframe, configure_ui, review_result, clean_result, include_outliers
from google.colab import data_table
data_table.enable_dataframe_formatter()

if not IN_COLAB or colab_files is None:
    raise RuntimeError("This upload cell is intended to run in Google Colab.")

uploaded = colab_files.upload()
if not uploaded:
    raise ValueError("Choose a .csv, .xls, or .xlsx file first.")

uploaded_filename, content = next(iter(uploaded.items()))
uploaded_dataframe = read_table(uploaded_filename, bytes(content))
configure_ui = None
review_result = None
clean_result = None
include_outliers = []

print(f"{uploaded_filename}: {len(uploaded_dataframe)} rows, {len(uploaded_dataframe.columns)} columns.")
rename_map = {
    col: f"col_{i}" for i, col in enumerate(uploaded_dataframe.columns)
}

df_view = uploaded_dataframe.rename(columns=rename_map)
col_map = pd.DataFrame({
    "shortened_column_name": list(rename_map.values()),
    "original_column_name": list(rename_map.keys())
})


display(data_table.DataTable(col_map,num_rows_per_page=5))
data_table.DataTable(df_view,num_rows_per_page=10)


In [ ]:
# @title 2. ⚙️ Customize configuration {"vertical-output":true,"display-mode":"form"}

global configure_ui


class ConfigureCleaningUI:
    def __init__(self, dataframe):
        self.dataframe = dataframe
        self.column_controls = []
        self.irb_text = widgets.Text(
            description="IRB(s)",
            placeholder="58807, 54909, OL, (you may enter multiple codes/keywords separated by ',')",
            layout=widgets.Layout(width="100%"),
        )
        self.subject_dropdown = widgets.Dropdown(
            description="Subject key identifier column",
            options=[str(column) for column in dataframe.columns],
            layout=widgets.Layout(width="100%"),
            style={"description_width": "190px"},
        )
        detected = detect_subject_column(dataframe.columns)
        if detected:
            self.subject_dropdown.value = detected
        self.drop_empty_rows = widgets.Checkbox(
            value=False,
            description="Drop fully empty rows",
            indent=False,
        )
        self.column_table = widgets.GridBox(
            [],
            layout=widgets.Layout(
                width="100%",
                max_width="none",
                max_height="520px",
                overflow="auto",
                border="1px solid #c9d6dd",
                padding="0",
                grid_template_columns="72px minmax(260px, 1fr) minmax(190px, 280px)",
                align_items="stretch",
            ),
        )
        self._render_column_controls()
        self.ui = widgets.VBox(
            [
                widgets.HTML("<h3>Configure cleaning</h3>"),
                widgets.VBox([self.irb_text, self.subject_dropdown, self.drop_empty_rows]),
                widgets.HTML(f"<b>Column keep / rename</b>: {len(dataframe.columns)} columns loaded."),
                self.column_table,
            ],
            layout=widgets.Layout(width="100%", max_width="none"),
        )

    @property
    def subject_column(self):
        return self.subject_dropdown.value

    @property
    def irbs(self):
        return [value.strip() for value in self.irb_text.value.split(",") if value.strip()]

    def column_rules(self):
        return [
            ColumnRule(original=original, keep=keep.value, rename=rename.value)
            for original, keep, rename in self.column_controls
        ]

    def _render_column_controls(self):
        cells = [
            widgets.HTML("<b>Keep</b>"),
            widgets.HTML("<b>Column</b>"),
            widgets.HTML("<b>Alternative name</b>"),
        ]
        for column in (str(column) for column in self.dataframe.columns):
            keep = widgets.Checkbox(value=True, indent=False, layout=widgets.Layout(width="34px"))
            rename = widgets.Text(placeholder="Optional new name", layout=widgets.Layout(width="100%"))
            label = widgets.HTML(f"<code>{column}</code>")
            self.column_controls.append((column, keep, rename))
            cells.extend([keep, label, rename])
        self.column_table.children = tuple(cells)


configure_ui = ConfigureCleaningUI(require_uploaded_dataframe())
display(configure_ui.ui)


In [ ]:
# @title 3. 🧐 Review subject ID exclusions {"vertical-output":true,"display-mode":"form"}

global review_result

review_result = run_clean(include_outlier_values=[])

if review_result.outliers.empty:
    print("No subject ID outliers found.")
else:
    print(f"{len(review_result.outliers)} subject IDs need review.")
    display(data_table.DataTable(review_result.outliers,num_rows_per_page=10))


In [ ]:
# @title 4a. 🗓️ Review visit/timepoint mapping {"vertical-output":true,"display-mode":"form"}

global timepoint_ui, timepoint_dictionary

timepoint_ui = TimepointMappingUI(require_uploaded_dataframe(), require_configure_ui())
timepoint_dictionary = timepoint_ui.timepoint_dictionary()
display(timepoint_ui.ui)


In [ ]:
# @title 5. 🧹🧼 Clean and preview {"vertical-output":true,"display-mode":"form"}

from IPython.display import HTML, display

global clean_result, timepoint_ui, timepoint_dictionary

if timepoint_ui is None:
    timepoint_ui = TimepointMappingUI(require_uploaded_dataframe(), require_configure_ui())

clean_result = run_clean(include_outlier_values=include_outliers)
timepoint_dictionary = apply_timepoint_mapping(clean_result, timepoint_ui)

print()
display(HTML("""
<h1 style="font-size:16px; margin: 0 0 16px 0;">
  Cleaned data previews:
</h1>
"""))

print(f"{len(clean_result.cleaned)} cleaned rows are ready. Showing first 100 rows.")

print()

display(data_table.DataTable(clean_result.cleaned,num_rows_per_page=10))
cleaned_data_summary(clean_result)

print('\n')
display(HTML("""
<h1 style="font-size:16px; margin: 0 0 16px 0;">
  Timepoint dictionary preview:
</h1>
"""))
print(f"{len(timepoint_dictionary)} rows")
print()


display(data_table.DataTable(timepoint_dictionary,num_rows_per_page=10))


print()
display(HTML("""
<h1 style="font-size:16px; margin: 0 0 16px 0;">
  Variable map previews:
</h1>
"""))
for sheet_name, dataframe in clean_result.variable_map.items():
    print()
    display(HTML(f"""
    <h1 style="font-size:16px; margin: 0 0 16px 0;">
      {sheet_name}
    </h1>
    """))
    print(f"{len(dataframe)} rows")
    display(data_table.DataTable(dataframe,num_rows_per_page=10))


In [ ]:
# @title 6. ⏬ Download cleaned files {"vertical-output":true,"display-mode":"form"}

global cleaned_path, map_path

if clean_result is None:
    raise ValueError("Run step 5 to clean and preview the file first.")

stem = output_stem(uploaded_filename)
cleaned_path = Path(f"{stem}_cleaned.xlsx")
map_path = None

if timepoint_dictionary is None:
    timepoint_dictionary = empty_timepoint_dictionary()

column_variable_dictionary = clean_result.variable_map.get(
    "column_map",
    pd.DataFrame(columns=["original_column", "clean_column", "keep", "role"]),
)
excluded_rows = clean_result.variable_map.get(
    "excluded_subjects",
    pd.DataFrame(columns=["raw_subid", "reason"]),
)

cleaned_path.write_bytes(
    dataframes_to_excel_bytes(
        {
            "_raw": uploaded_dataframe,
            "_cleaned": clean_result.cleaned,
            "timepoint_dictionary": timepoint_dictionary,
            "column_variable_dictionary": column_variable_dictionary,
            "excluded_rows": excluded_rows,
        }
    )
)


filename = input(f'\n\n ✏️ Please enter the file name to save this run. \n\nDefault is {stem}_cleaned.xlsx\n')
if len(filename) < 2: filename = cleaned_path
if '.xlsx' not in filename: filename += '.xlsx'
print(f'\n\n📓 Downloading {filename}')


download_path = Path(filename)
download_path.write_bytes(cleaned_path.read_bytes())


if IN_COLAB:
    colab_files.download(str(download_path))
else:
    display(FileLink(str(cleaned_path)))
